[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/kasparvonbeelen/contracts/blob/main/2-4-classify-apply_models_seq.ipynb)


# Apply Classifier

In [ ]:
!pip install -q -U datasets transformers evaluate

In [ ]:
import ast
import json
import numpy as np
import torch
from torch.nn.functional import softmax
from pathlib import Path
import pandas as pd
from tqdm.auto import tqdm
from transformers import AutoModelForSequenceClassification, AutoTokenizer

In [ ]:
# metadata.tsv is in the repository as a zip (e.g. on Colab); unzip it only if it is not there yet
import zipfile
if not Path('processed_data/tous/metadata.tsv').exists():
    zip_path = next(p for p in [Path('processed_data/tous/metadata.tsv.zip'), Path('metadata.tsv.zip')] if p.exists())
    zipfile.ZipFile(zip_path).extractall('processed_data/tous')

In [2]:
processed_data_dir = Path('processed_data/tous')
file_name = processed_data_dir / 'metadata.tsv' # _annotated_distilbert
metadata = pd.read_csv(file_name,sep='\t')
metadata.fillna('', inplace=True)
print(len(metadata))

124495


In [3]:
metadata.columns

Index(['platform', 'year', 'sentence', 'sentence_processed'], dtype='object')

In [ ]:
clause_type = 'modification' # 'modification' | 'arbitration' | 'anti-scraping'

# The model trained by 2-3-classify-train_models_seq.ipynb on annotations/claude_annotations_combined.
# 2-3 saves a sentence-only model (its default) to models/<clause>_best_model_modernbert_sentence and a
# context model to models/<clause>_best_model_modernbert.
MODEL_DIR = Path(f'models/{clause_type}_best_model_modernbert_sentence')

# USE_CONTEXT / CONTEXT_WINDOW are read from classifier_config.json, written by 2-3 next to the model,
# so the input is built exactly as in training. The values below are only used if that file is missing.
USE_CONTEXT = False
CONTEXT_WINDOW = 2
config_file = MODEL_DIR / 'classifier_config.json'
if config_file.exists():
    classifier_config = json.loads(config_file.read_text())
    USE_CONTEXT, CONTEXT_WINDOW = classifier_config['use_context'], classifier_config['context_window']
    assert classifier_config['clause_type'] == clause_type, f"{MODEL_DIR} was trained for {classifier_config['clause_type']}"
    print('trained on', classifier_config['annotation_file'], '| test metrics:',
          {k: round(v, 3) for k, v in classifier_config['eval'].items() if k in ('eval_f1', 'eval_precision', 'eval_recall')})
else:
    print(f'{config_file} not found: using USE_CONTEXT = {USE_CONTEXT}, CONTEXT_WINDOW = {CONTEXT_WINDOW}')

THRESHOLD = 0.5      # label = prob >= THRESHOLD; see 2-6-classify_evaluate.ipynb for the choice per clause type

# judge-annotated training data, used below to flag which corpus sentences were in the training set
ANNOTATION_DIR = Path('annotations/claude_annotations_combined')

model = AutoModelForSequenceClassification.from_pretrained(MODEL_DIR)
tokenizer = AutoTokenizer.from_pretrained(MODEL_DIR)  # a context model's tokenizer already knows the [TGT] token
if USE_CONTEXT != ('[TGT]' in tokenizer.get_vocab()):
    raise ValueError(f'{MODEL_DIR}: USE_CONTEXT = {USE_CONTEXT} does not match the model '
                     f"({'has' if '[TGT]' in tokenizer.get_vocab() else 'has no'} [TGT] token)")

device = (
    "cuda" if torch.cuda.is_available()          # Colab GPU
    else "mps" if torch.backends.mps.is_available()  # macbook
    else "cpu"
)
model.to(device)
model.eval()
print(device, '| context' if USE_CONTEXT else '| target sentence only', '|', MODEL_DIR)

In [5]:
metadata.head()

,platform,year,sentence,sentence_processed
0,twitter,20150501,Twitter Terms of Service These Terms of Servic...,[mask] [mask] [mask] [mask] [mask] [mask] [mas...
1,twitter,20150501,Your access to and use of the Services are con...,your access to and use of the services are con...
2,twitter,20150501,By accessing or using the Services you agree t...,by accessing or using the services you agree t...
3,twitter,20150501,Basic Terms You are responsible for your use o...,basic terms you are responsible for your use o...
4,twitter,20150501,"Most Content you submit, post, or display thro...","most content you submit , post , or display th..."


In [ ]:
# With context, the same sentence text can get a different prediction depending on where it sits in a
# document, so every row gets its own '[TGT]'-wrapped input. Without context, the input is the sentence
# itself, so each distinct sentence is classified once and the result is mapped back to all its rows.
metadata['tou_id'] = metadata['platform'].astype(str) + '_' + metadata['year'].astype(str)

sentences = metadata['sentence'].fillna('').tolist()
tou_ids = metadata['tou_id'].tolist()
n = len(metadata)

def build_context_text(i, window=CONTEXT_WINDOW):
    """Mirrors build_text (2-3): wraps the target sentence in [TGT] ... [TGT] inside up to
    `window` sentences of context on each side, never crossing into a different tou_id."""
    tid = tou_ids[i]
    prev = [sentences[j] for j in range(max(0, i - window), i) if tou_ids[j] == tid]
    nxt = [sentences[j] for j in range(i + 1, min(n, i + 1 + window)) if tou_ids[j] == tid]
    return f"{' '.join(prev)} [TGT] {sentences[i]} [TGT] {' '.join(nxt)}".strip()

if USE_CONTEXT:
    metadata['text'] = [build_context_text(i) for i in tqdm(range(n))]
else:
    metadata['text'] = metadata['sentence'].fillna('')

texts_to_classify = metadata['text'].drop_duplicates().tolist()
print(f"{len(texts_to_classify)} inputs to classify for {n} sentences")

In [ ]:
batch_size = 64 if device == "cuda" else 16  # bigger batches on a Colab GPU; smaller on mps/cpu

probs = []
with torch.no_grad():
    for i in tqdm(range(0, len(texts_to_classify), batch_size)):
        batch_texts = texts_to_classify[i:i + batch_size]
        inputs = tokenizer(batch_texts, return_tensors='pt', truncation=True, padding=True).to(device)
        logits = model(**inputs).logits
        probs.extend(softmax(logits.float(), dim=1)[:, 1].cpu().tolist())

metadata[f'prob_1_{clause_type}'] = metadata['text'].map(dict(zip(texts_to_classify, probs)))

In [ ]:
metadata[clause_type] = (metadata[f'prob_1_{clause_type}'] >= THRESHOLD).astype(int)

In [ ]:
# separate output files for the context and the sentence-only classifier
output_file_name = processed_data_dir / f"metadata_annotated_modernbert{'' if USE_CONTEXT else '_sentence'}.tsv"
output_file_name

In [ ]:
metadata.to_csv(output_file_name, sep='\t', index=False)

In [ ]:
metadata[clause_type].value_counts()

In [ ]:
metadata.columns

In [ ]:
metadata['year_int'] = metadata.year.apply(lambda x: int(str(x)[:4]))

In [ ]:
import seaborn as sns

data = metadata.groupby(['platform','year_int'])[clause_type].sum().astype(bool).astype(int).unstack().fillna(-1)



#.loc['bumble'].plot(kind='bar')

In [ ]:
import pandas as _pd

def replace_minus_ones_with_prev(X, axis=1, inplace=False):
    """
    Replace -1 entries in a matrix/array with the nearest preceding 0 or 1 along the given axis.
    If there is no preceding non -1 value, the -1 is left unchanged.

    Parameters:
    - X: array-like (numpy array, list of lists, or pandas DataFrame)
    - axis: 1 to replace along rows (left-to-right), 0 to replace along columns (top-to-bottom)
    - inplace: if True and X is a numpy array or DataFrame, modify it in place; otherwise return a new array

    Returns:
    - numpy.ndarray or pandas.DataFrame with replacements applied (unless inplace=True modifies input)
    """


    is_df = _pd is not None and isinstance(X, _pd.DataFrame)
    if is_df:
        arr = X.values
    else:
        arr = X if isinstance(X, (np.ndarray,)) else np.array(X)

    if not inplace:
        arr = arr.copy()

    if axis not in (0, 1):
        raise ValueError("axis must be 0 or 1")

    # iterate over the chosen axis and carry forward the last seen non -1 value
    if axis == 1:
        # rows
        for r in range(arr.shape[0]):
            last = None
            for c in range(arr.shape[1]):
                val = arr[r, c]
                if val != -1:
                    last = val
                elif last is not None:
                    arr[r, c] = last
    else:
        # columns
        for c in range(arr.shape[1]):
            last = None
            for r in range(arr.shape[0]):
                val = arr[r, c]
                if val != -1:
                    last = val
                elif last is not None:
                    arr[r, c] = last

    if is_df:
        if inplace:
            X.iloc[:, :] = arr
            return X
        else:
            return _pd.DataFrame(arr, index=X.index, columns=X.columns)
    else:
        return arr

sns.set(rc={'figure.figsize':(6.7,10.27)})
sns.heatmap(replace_minus_ones_with_prev(data),cbar=False)

In [ ]:
metadata.columns

In [ ]:
#metadata.drop(columns=['logits_arbitration', 'logits_anti-scraping'], inplace=True)

In [ ]:
metadata.columns

In [ ]:
#metadata.sort_values('prob_1', ascending=False).head(10)

In [ ]:
# load the judge-annotated training data (annotations/claude_annotations_combined, built at the end of
# 2-6-classify_evaluate.ipynb) to flag which corpus sentences were annotated
df_annotations = pd.read_csv(ANNOTATION_DIR / f'{clause_type}_annotations.csv')
annotation_label_col = [c for c in df_annotations.columns if c.startswith('label_')][0]
# `id` is the row of the sentence in the metadata; flag rows, and (below) sentence texts
metadata['annotated'] = metadata.index.isin(df_annotations['id'])
print(f"{len(df_annotations)} annotations ({df_annotations[annotation_label_col].value_counts().to_dict()}), "
      f"{metadata['annotated'].sum()} metadata rows flagged as annotated")

In [ ]:
df_deduplicated = metadata.drop_duplicates(subset=['sentence']).copy()
df_deduplicated['annotated'] = df_deduplicated.sentence.isin(df_annotations.Target_sentence)
int_labels = [((0.95,1.0),'confident_positive'),( (0.80,.95), 'sure_positive'),((0.60,.80), 'leaning_positive'),
                   ((0.50,.60), 'borderline_positive'),((0.40,.50), 'borderline_negative'),
                   ((0.20,.40), 'leaning_negative'),((0.05,.20), 'sure_negative'),((0.0,.05), 'confident_negative')]
for interval, label in int_labels:
    df_deduplicated.loc[df_deduplicated[f'prob_1_{clause_type}'].between(*interval),'category']  = label
df_deduplicated['category'].value_counts()

In [ ]:
Path('annotations/inference').mkdir(parents=True, exist_ok=True)
suffix = '' if USE_CONTEXT else '_sentence'
# up to 10 sentences per category (some categories can have fewer)
pd.concat([df_deduplicated[df_deduplicated.category == label].pipe(lambda d: d.sample(min(10, len(d)), random_state=42))
    for _ , label in int_labels], axis=0)[['sentence','category', 'annotated']].to_csv(
        f'annotations/inference/{clause_type}_automatic_annotations_by_category{suffix}.csv')

In [ ]:
metadata[metadata[clause_type] == 1].to_csv(f'annotations/inference/{clause_type}_inference{suffix}.csv')

## Fin